In [22]:
import os  #It helps with 1.Looking for folders 2.Creating Files 3.Deleting Files 4.Checking path to your computer
import time  #gives Python tools to pause execution, get the current time, and measure how long operations take.
import torch  #This is helpful for trainning AI engine loads the PyTorch library, which provides tensors, fast math operations, and tools for building and training deep learning models
import torch.nn as nn  #Helps to build Neural Netwrok imports PyTorch’s neural network module and gives it the short name nn so we can easily build neural network layers.
import torch.nn.functional as F  #provides individual neural network functions like ReLU, softmax, and loss calculations
import torch.optim as optim  #provides algorithms that update model weights to reduce prediction errors during training.
from torchvision import datasets, transforms, models  #dataset Ready-made training data   Transforms->Transforms modify images before training.
from torch.utils.data import DataLoader, random_split #DataLoader feeds data to the model in batches -- random_split divides datasets into training and validation sets.
from tqdm.notebook import tqdm  #Shows the model processing



In [ ]:
from google.colab import drive  # Our data is located at the google drive as the google colab has temporary file storage
drive.mount('/content/drive')

# Set the device to CPU  -- GPU is faster than CPU in processing the images and specialized in doing math Can do thousands of calcuations and perfect for deep learning
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device.")

# 3. Set your data directory
# UPDATE THIS PATH if your 'Data' folder is nested somewhere else in your Drive
data_dir = "/content/drive/MyDrive/Data/train"
print("Data Directory:", data_dir)

In [ ]:
#This class have a construcotr the check if the image format is RGB if not it convert it to RGB it helps to normalize the data
class ConvertToRGB(object):
    def __call__(self, img):
        if img.mode != "RGB":  #if not RGB format - convert it to RGB
            img = img.convert("RGB")
        return img

# ----Creating Image Normalization -----
#This is going to conver to RGB so it makes it colorful
#its going to resize the images so they are all in same size
#ToTensor converts an image into a PyTorch tensor and scales pixel values from 0–255 to 0–1 so the neural network can process the image.
transform_normalized = transforms.Compose([
    ConvertToRGB(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Load the datasets directly from your drive
# Passing the data_dir to the datasets.ImageFolder which have two attributes root(where the file is) transform(transform function to convert the image to standardized format)
try:
    normalized_dataset = datasets.ImageFolder(root=data_dir, transform=transform_normalized)
    print(f"Success! Found {len(normalized_dataset)} images belonging to {len(normalized_dataset.classes)} classes.")
    print("Classes:", normalized_dataset.classes)
except Exception as e:
    print("Error loading data. Please double-check your data_dir path in Cell 1!")
    print(e)


In [ ]:
#This class create a neurl network architecture -- Our class is extended from nn.Module  -- nn.Module alraedy know how to store layers , track parameters , move model to gpu , run forward passes , save load models
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):  #The model predict 5 categories 4 diesaese and 1 helthy
        super(SimpleCNN, self).__init__()  #This is initilizing everything from the nn.Module parent class -- nn.Module is the parent class.
        # 1st Convolutional Layer
        #Convolution layer is the layer where scan the images with small filters to detect patterns and produce feature maps used for image recognition
        #Using Conv2d because it has two dimension here heigh and width
        #in_channels = 3 tells how many channels the input image has here is Red Green Blue
        #out_channels=16 tells how many filters it will learn filter 1-> vertical edges , filter 2 -> horizental edges and ...
        #kernel_size=3 this defines the size of the filter the filters slides across the images and check small region --> (3 * 3 magnifying glass)
        #padding = 1 Padding preserves the image size after convolution.
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        #Max Pooling is looking at small regions and keeping only the stronges signal
        # Kernel size = 2 2 * 2 find the biggest number
        #Stride says what is the next step would be if its [0-1] ->[2-3]->[4-5]
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 2nd Convolutional Layer
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Fully Connected Layers
        #Flattening the images - convert 3d images to one long vector
        self.fc1 = nn.Linear(32 * 56 * 56, 128)    # 32 * 56 * 56 input -- 128 output
        self.fc2 = nn.Linear(128, num_classes)  # conver the 128 outoput of previous layer to 5 disease classes

        #Forward function sends the entry images through the machines in this exact order
        #Defines how data flows through the network
        # Relu ->Rectified Linear Unit --> its activation function that repalces negative values with zero allowing neural network to learn complex nonlinear patterns
        #Activation function transfroms a neuron's output using a nonlinear rule so the neural network can learn complex pattern

    def forward(self, x):
      x = self.pool(F.relu(self.conv1(x)))  #self.pool shrink the image while keeping the most importnat vlaues
      x = self.pool(F.relu(self.conv2(x)))
      x = x.view(-1, 32 * 56 * 56) # Flatten
      x = F.relu(self.fc1(x))
      x = self.fc2(x)
      return x

# Test if the model builds and can be moved to the GPU
model_simple = SimpleCNN().to(device)
print("Method 2 (Simple CNN) built successfully!")

In [ ]:
!git clone https://github.com/EHSANHAZARI/Uganda-Crop-Disaese.git

In [ ]:
#Train_epoch teaches the neural network by processing all trainning batches once, calculating the error and updating the model weight
#Model is the neural network we want to train
# loss_fn calcutes how wrong the prediction is : example nn.CrossEntropyLoss()
#Data_loader provied the datset in the batches
# device tells the model where to run
def train_epoch(model, optimizer, loss_fn, data_loader, device="cpu"):
    model.train()
    training_loss = 0.0
    # Going through the datasets batch by batch get the image and the correct label for it
    # input --> image , output --> correct label
    for inputs, targets in data_loader:
      #we have to make sure input and target are both on the same machine that's what .to(device) do
        inputs, targets = inputs.to(device), targets.to(device)
        #.zero_grad clears the stored gradients(mistakes) before calculating new ones.
        optimizer.zero_grad()
        #Send the input data (images) into the neural network and get the prediction.
        output = model(inputs)
        # Calcuating the loss using the output and label (targets)
        loss = loss_fn(output, targets)
        #Compute gradients   loss.backward()-> Which weights caused this mistake?
        loss.backward()
        #Update weights
        optimizer.step()
        #loss.data.item() --> This converts the tensor into a normal Python number
        #inputs.size(0) --> gives the batch size
        training_loss += loss.data.item() * inputs.size(0)
    #Finding the average mistake and return it
    return training_loss / len(data_loader)
def score(model, data_loader, loss_fn, device="cpu"):
    model.eval()
    total_loss = 0
    total_correct = 0
    n_observations = 0



In [ ]:
!git add .

In [ ]:
!git branch


In [ ]:
!git clone https://github.com/EHSANHAZARI/Uganda-Crop-Disaese.git

In [ ]:
!ls
!git status

In [ ]:
%cd Uganda-Crop-Disaese

In [ ]:
!ls
!git status

In [ ]:
%cd /content/Uganda-Crop-Disaese
!pwd
!ls

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/AIProject.ipynb" /content/Uganda-Crop-Disaese/AIProject.ipynb